# Uncertainty-Aware Skin Lesion Segmentation

**Bayesian U-Net with MC Dropout on ISIC 2018**

## Setup
1. Click **+ Add Data** (right panel) &rarr; search **ISIC2018 Challenge Task1 Data Segmentation** &rarr; add dataset by **tschandl**
2. Settings &rarr; Accelerator &rarr; **GPU T4 x2** (or P100)
3. Run all cells in order

Training takes ~15-25 min on a T4 GPU.

In [ ]:
!pip install -q segmentation-models-pytorch

In [ ]:
import os, csv, io, time, random
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp
import cv2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Enable GPU in Settings > Accelerator")

In [ ]:
IMAGE_SIZE = 256
BATCH_SIZE = 32
EPOCHS = 100
LR = 1e-4
MC_PASSES = 20
WORKING = Path("/kaggle/working")
SPLITS = WORKING / "splits"

# --- Discover ISIC 2018 dataset paths ---
KAGGLE_INPUT = Path("/kaggle/input")

print("Datasets in /kaggle/input/:")
for d in sorted(KAGGLE_INPUT.iterdir()):
    print(f"  {d.name}/")

images_dir = masks_dir = None
for root, dirs, files in os.walk(KAGGLE_INPUT):
    root_path = Path(root)
    jpgs = [f for f in files if f.lower().endswith('.jpg')]
    pngs = [f for f in files if f.lower().endswith('.png') and '_segmentation' in f.lower()]
    if jpgs and len(jpgs) > 100 and images_dir is None:
        images_dir = root_path
    if pngs and len(pngs) > 100 and masks_dir is None:
        masks_dir = root_path

if not images_dir or not masks_dir:
    raise FileNotFoundError(
        "Could not find ISIC image/mask folders. "
        "Make sure the dataset contains Training_Input and Training_GroundTruth folders."
    )
n_img = len(list(images_dir.glob('*.jpg')))
n_msk = len(list(masks_dir.glob('*.png')))
print(f"Images: {images_dir} ({n_img} files)")
print(f"Masks:  {masks_dir} ({n_msk} files)")

# --- Create train / val / test splits (70/15/15, seed=42) ---
image_files = sorted(f.stem for f in images_dir.glob("*.jpg"))
mask_ids = {f.stem.replace("_segmentation", "") for f in masks_dir.glob("*.png")}
image_ids = sorted([iid for iid in image_files if iid in mask_ids])
print(f"Matched image-mask pairs: {len(image_ids)}")

rng = random.Random(42)
indices = list(range(len(image_ids)))
rng.shuffle(indices)

n = len(image_ids)
n_train = int(n * 0.70)
n_val = (n - n_train) // 2

train_ids = sorted([image_ids[i] for i in indices[:n_train]])
val_ids   = sorted([image_ids[i] for i in indices[n_train:n_train + n_val]])
test_ids  = sorted([image_ids[i] for i in indices[n_train + n_val:]])

SPLITS.mkdir(parents=True, exist_ok=True)
for name, ids in [("train", train_ids), ("val", val_ids), ("test", test_ids)]:
    with open(SPLITS / f"{name}.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["image_id"])
        for iid in ids:
            w.writerow([iid])

print(f"Splits: {len(train_ids)} train | {len(val_ids)} val | {len(test_ids)} test")

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)


class ISICDataset(Dataset):
    def __init__(self, split, image_size=IMAGE_SIZE, augment=False):
        csv_path = SPLITS / f"{split}.csv"
        self.image_ids = []
        with open(csv_path) as f:
            for row in csv.DictReader(f):
                self.image_ids.append(row["image_id"])
        self.image_size = image_size
        self.augment = augment and (split == "train")

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        iid = self.image_ids[idx]
        img  = Image.open(images_dir / f"{iid}.jpg").convert("RGB")
        mask = Image.open(masks_dir  / f"{iid}_segmentation.png").convert("L")

        img  = img.resize((self.image_size, self.image_size), Image.BILINEAR)
        mask = mask.resize((self.image_size, self.image_size), Image.NEAREST)

        img  = np.array(img,  dtype=np.float32) / 255.0
        mask = np.array(mask, dtype=np.float32) / 255.0
        mask = (mask > 0.5).astype(np.float32)

        if self.augment:
            rng = np.random.default_rng()
            if rng.random() > 0.5:
                img  = np.flip(img,  axis=1).copy()
                mask = np.flip(mask, axis=1).copy()
            if rng.random() > 0.5:
                img  = np.flip(img,  axis=0).copy()
                mask = np.flip(mask, axis=0).copy()
            k = rng.integers(0, 4)
            if k > 0:
                img  = np.rot90(img,  k, axes=(0, 1)).copy()
                mask = np.rot90(mask, k, axes=(0, 1)).copy()

        img = (img - IMAGENET_MEAN) / IMAGENET_STD
        img  = torch.from_numpy(img.transpose(2, 0, 1))
        mask = torch.from_numpy(mask[np.newaxis, :, :])
        return img, mask


train_ds = ISICDataset("train", augment=True)
val_ds   = ISICDataset("val",   augment=False)
print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, persistent_workers=True,
                          prefetch_factor=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, persistent_workers=True,
                          prefetch_factor=2, pin_memory=True)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

In [ ]:
class DropoutUnet(nn.Module):
    def __init__(self, p=0.3):
        super().__init__()
        self.model = smp.Unet(
            encoder_name="resnet34",
            encoder_weights="imagenet",
            in_channels=3, classes=1,
        )
        self.dropout = nn.Dropout2d(p=p)

    def forward(self, x):
        features = self.model.encoder(x)
        features[-1] = self.dropout(features[-1])
        decoder_output = self.model.decoder(features)
        return self.model.segmentation_head(decoder_output)


def get_unet():
    return DropoutUnet(p=0.3)


def enable_dropout(model):
    for m in model.modules():
        if isinstance(m, (nn.Dropout, nn.Dropout2d)):
            m.train()


# --- Segmentation metrics ---

def dice_score(pred, target, eps=1e-7):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    inter = (pred * target).sum(dim=(1, 2, 3))
    union = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    return ((2.0 * inter + eps) / (union + eps)).mean()


def iou_score(pred, target, eps=1e-7):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    inter = (pred * target).sum(dim=(1, 2, 3))
    union = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3)) - inter
    return ((inter + eps) / (union + eps)).mean()


def dice_loss(logits, targets, eps=1e-7):
    probs = torch.sigmoid(logits)
    inter = (probs * targets).sum(dim=(1, 2, 3))
    union = probs.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
    return 1 - ((2.0 * inter + eps) / (union + eps)).mean()


# --- Uncertainty metrics ---

def predictive_entropy(mean_prob, eps=1e-7):
    return -(mean_prob * torch.log(mean_prob + eps) +
             (1 - mean_prob) * torch.log(1 - mean_prob + eps))


def expected_entropy(samples, eps=1e-7):
    ent = -(samples * torch.log(samples + eps) +
            (1 - samples) * torch.log(1 - samples + eps))
    return ent.mean(dim=0)


def mutual_information(mean_prob, samples):
    return predictive_entropy(mean_prob) - expected_entropy(samples)


# --- Calibration ---

def pixel_ece(all_probs, all_labels, n_bins=15):
    boundaries = np.linspace(0, 1, n_bins + 1)
    accs   = np.zeros(n_bins)
    confs  = np.zeros(n_bins)
    counts = np.zeros(n_bins)
    for i in range(n_bins):
        lo, hi = boundaries[i], boundaries[i + 1]
        if i == n_bins - 1:
            mask = (all_probs >= lo) & (all_probs <= hi)
        else:
            mask = (all_probs >= lo) & (all_probs < hi)
        counts[i] = mask.sum()
        if counts[i] > 0:
            confs[i] = all_probs[mask].mean()
            accs[i] = all_labels[mask].mean()
    weights = counts / max(all_probs.shape[0], 1)
    ece = (weights * np.abs(accs - confs)).sum()
    return ece, accs, confs, counts


print("Model, metrics, and utilities defined.")

In [ ]:
def train_one_epoch(model, loader, optimizer, bce_fn, epoch, total_epochs):
    model.train()
    total_loss = 0.0
    pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{total_epochs} [train]", leave=False)
    for i, (images, masks) in enumerate(pbar):
        images = images.to(device, non_blocking=True)
        masks  = masks.to(device, non_blocking=True)
        with torch.autocast("cuda"):
            logits = model(images)
            loss = bce_fn(logits, masks) + dice_loss(logits, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        pbar.set_postfix(loss=f"{total_loss / (i + 1):.4f}")
    return total_loss / len(loader)


def validate(model, loader, epoch, total_epochs):
    model.eval()
    total_dice, total_iou = 0.0, 0.0
    pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{total_epochs} [val]", leave=False)
    with torch.no_grad(), torch.autocast("cuda"):
        for images, masks in pbar:
            images = images.to(device, non_blocking=True)
            masks  = masks.to(device, non_blocking=True)
            logits = model(images)
            d = dice_score(logits, masks).item()
            total_dice += d
            total_iou  += iou_score(logits, masks).item()
            pbar.set_postfix(dice=f"{d:.4f}")
    n = len(loader)
    return total_dice / n, total_iou / n


print("Training functions defined.")

In [ ]:
model = get_unet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
bce_fn = nn.BCEWithLogitsLoss()

best_dice = 0.0
checkpoint_path = WORKING / "best_model.pth"
history = {"train_loss": [], "val_dice": [], "val_iou": []}

print(f"Training: {EPOCHS} epochs | Batch: {BATCH_SIZE} | LR: {LR}")
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")
print()

for epoch in range(EPOCHS):
    t0 = time.time()
    train_loss = train_one_epoch(model, train_loader, optimizer, bce_fn, epoch, EPOCHS)
    val_dice, val_iou = validate(model, val_loader, epoch, EPOCHS)
    elapsed = time.time() - t0

    history["train_loss"].append(train_loss)
    history["val_dice"].append(val_dice)
    history["val_iou"].append(val_iou)

    star = ""
    if val_dice > best_dice:
        best_dice = val_dice
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_dice": val_dice, "val_iou": val_iou,
        }, str(checkpoint_path))
        star = " *best*"

    print(f"Epoch {epoch+1:3d}/{EPOCHS} | Loss: {train_loss:.4f} | "
          f"Dice: {val_dice:.4f} | IoU: {val_iou:.4f} | {elapsed:.0f}s{star}")

print()
print(f"Best Dice: {best_dice:.4f} | Checkpoint: {checkpoint_path}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history["train_loss"], label="Train Loss", color="tab:red")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.set_title("Training Loss")
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history["val_dice"], label="Dice", color="tab:blue")
ax2.plot(history["val_iou"],  label="IoU",  color="tab:green")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Score"); ax2.set_title("Validation Metrics")
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(WORKING / "training_curves.png"), dpi=150, bbox_inches="tight")
plt.show()

## MC Dropout Evaluation

Run T=20 stochastic forward passes with dropout enabled at test time to estimate:
- **Predictive entropy** &mdash; total uncertainty
- **Mutual information** &mdash; epistemic (model) uncertainty
- **ECE** &mdash; calibration error

In [ ]:
model = get_unet().to(device)
ckpt = torch.load(str(checkpoint_path), map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Loaded: epoch {ckpt['epoch']}, Dice {ckpt['val_dice']:.4f}")

test_ds = ISICDataset("test", augment=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=4, pin_memory=True)
print(f"Test set: {len(test_ds)} images")

model.eval()
enable_dropout(model)

all_dice, all_iou, all_mi = [], [], []
all_probs_flat, all_labels_flat = [], []

for images, masks in tqdm(test_loader, desc="MC Eval"):
    images, masks = images.to(device), masks.to(device)
    preds = []
    for _ in range(MC_PASSES):
        with torch.no_grad():
            logits = model(images)
            probs = torch.sigmoid(logits)
            preds.append(probs.cpu())

    samples = torch.stack(preds)
    mean_pred = samples.mean(dim=0).to(device)
    logits_eq = torch.logit(mean_pred.clamp(1e-6, 1 - 1e-6))

    all_dice.append(dice_score(logits_eq, masks).item())
    all_iou.append(iou_score(logits_eq, masks).item())

    mi = mutual_information(mean_pred, samples.to(device))
    all_mi.append(mi.mean().item())

    all_probs_flat.append(mean_pred.cpu().numpy().flatten())
    all_labels_flat.append(masks.cpu().numpy().flatten())

all_probs_flat = np.concatenate(all_probs_flat)
all_labels_flat = np.concatenate(all_labels_flat)
ece_val, bin_acc, bin_conf, bin_counts = pixel_ece(all_probs_flat, all_labels_flat)

print()
print("=" * 50)
print(f"  Test Results (MC Dropout, T={MC_PASSES})")
print("=" * 50)
print(f"  Dice : {np.mean(all_dice):.4f}")
print(f"  IoU  : {np.mean(all_iou):.4f}")
print(f"  ECE  : {ece_val:.4f}")
print(f"  MI   : {np.mean(all_mi):.6f}")
print("=" * 50)

In [ ]:
# --- Reliability Diagram ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6, 5),
                                gridspec_kw={"height_ratios": [3, 1]}, sharex=True)
n_bins = len(bin_acc)
centers = np.linspace(1/(2*n_bins), 1 - 1/(2*n_bins), n_bins)
width = 1.0 / n_bins

ax1.bar(centers, bin_acc, width=width*0.9, color="steelblue",
        edgecolor="black", linewidth=0.5, alpha=0.85, label="Accuracy")
ax1.plot([0, 1], [0, 1], "k--", linewidth=1, label="Perfect calibration")
ax1.set_ylabel("Accuracy")
ax1.set_title(f"Reliability Diagram (ECE = {ece_val:.4f})")
ax1.legend(loc="upper left", fontsize=8)
ax1.set_xlim(0, 1); ax1.set_ylim(0, 1)

ax2.bar(centers, bin_counts, width=width*0.9, color="gray",
        edgecolor="black", linewidth=0.5, alpha=0.6)
ax2.set_xlabel("Confidence"); ax2.set_ylabel("Count")
plt.tight_layout()
plt.savefig(str(WORKING / "reliability_diagram.png"), dpi=150, bbox_inches="tight")
plt.show()

# --- Sample predictions with uncertainty ---
model.eval()
enable_dropout(model)

for sample_idx in range(min(3, len(test_ds))):
    img, mask = test_ds[sample_idx]
    img_batch = img.unsqueeze(0).to(device)

    preds = []
    for _ in range(MC_PASSES):
        with torch.no_grad():
            logits = model(img_batch)
            probs = torch.sigmoid(logits)
            preds.append(probs.cpu())

    samples = torch.stack(preds)
    mean_pred = samples.mean(dim=0)
    epistemic = mutual_information(mean_pred, samples)
    total_unc = predictive_entropy(mean_pred)

    img_vis = img.numpy().transpose(1, 2, 0)
    img_vis = img_vis * IMAGENET_STD + IMAGENET_MEAN
    img_vis = np.clip(img_vis, 0, 1)

    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    axes[0].imshow(img_vis);                                     axes[0].set_title("Input")
    axes[1].imshow(mask[0], cmap="gray");                        axes[1].set_title("Ground Truth")
    axes[2].imshow(mean_pred[0, 0].numpy(), cmap="gray", vmin=0, vmax=1)
    axes[2].set_title("Mean Prediction")
    axes[3].imshow(epistemic[0, 0].numpy(), cmap="hot");         axes[3].set_title("Epistemic (MI)")
    axes[4].imshow(total_unc[0, 0].numpy(), cmap="hot");         axes[4].set_title("Total Uncertainty")
    for ax in axes:
        ax.axis("off")
    plt.suptitle(f"Sample {sample_idx + 1}: MC Dropout Uncertainty (T={MC_PASSES})", fontweight="bold")
    plt.tight_layout()
    plt.savefig(str(WORKING / f"uncertainty_sample_{sample_idx}.png"), dpi=150, bbox_inches="tight")
    plt.show()

## Download the Trained Model

The checkpoint (`best_model.pth`) is saved in `/kaggle/working/`.

**To use it locally:**
1. Download `best_model.pth` from the **Output** tab (right panel)
2. Copy it to your project's `src/` directory
3. Run evaluation locally:

```bash
cd src
python eval.py --checkpoint best_model.pth --use_isic --mc_passes 20
```

All plots and results are also available in the Output tab.